## 📊 Classificador de Planilhas — versão 2
**Preencha os campos abaixo e clique no botão ▶ para rodar.**

> As planilhas serão classificadas automaticamente e enviadas
> para o e-mail informado. Nenhuma alteração manual é necessária.
>
> A senha de app do Gmail **não** fica neste notebook: ela é pedida
> quando o processamento começa, ou lida dos *Secrets* do Colab
> (ícone de chave na barra lateral, com o nome `SENHA_APP`).

In [ ]:
#@title ⚙️ Configurações — preencha os campos e clique em ▶
#@markdown ### 📂 Google Drive
PASTA_DRIVE = "TESTE" #@param {type:"string"}

#@markdown ### 📧 E-mail de envio
EMAIL_REMETENTE = "seuemail@gmail.com" #@param {type:"string"}

#@markdown ### 📨 E-mail de destino
EMAIL_DESTINATARIO = "destinatario@email.com" #@param {type:"string"}

In [ ]:
import pandas as pd
import unicodedata
import openpyxl
import smtplib
import os
import re
import getpass
from copy import copy
from email.message import EmailMessage
from google.colab import drive
from openpyxl.utils import get_column_letter, range_boundaries

print("=" * 50)
print("  CLASSIFICADOR DE PLANILHAS — versão 2")
print("=" * 50)

# --- Conectar ao Drive ---
print("\n📂 Conectando ao Google Drive...")
drive.mount('/content/drive', force_remount=True)

caminho_pasta = f"/content/drive/MyDrive/{PASTA_DRIVE}"

if not os.path.exists(caminho_pasta):
    print(f"\n❌ ERRO: Pasta '{PASTA_DRIVE}' não encontrada no Drive.")
    print("Verifique o nome da pasta no campo de configuração.")
    raise SystemExit()

# --- Buscar planilhas nas subpastas ---
# Percorre tudo que estiver dentro de PASTA_DRIVE.
# Ignora PDFs, TXTs e qualquer .xlsx que não tenha
# a coluna "Discriminação".
print(f"\n🔍 Verificando conteúdo da pasta '{PASTA_DRIVE}'...")

arquivos_encontrados = []  # (caminho_completo, nome_subpasta, nome_arquivo)

for item in sorted(os.listdir(caminho_pasta)):
    caminho_item = os.path.join(caminho_pasta, item)

    if not os.path.isdir(caminho_item):
        continue  # Ignora arquivos soltos na raiz

    print(f"\n  📁 {item}/")

    for arquivo in sorted(os.listdir(caminho_item)):
        if not arquivo.endswith('.xlsx'):
            print(f"     ⏭️  {arquivo} (ignorado)")
            continue

        caminho_arquivo = os.path.join(caminho_item, arquivo)

        try:
            df_teste = pd.read_excel(caminho_arquivo, nrows=0)
            if "Discriminação" not in df_teste.columns:
                print(f"     ⏭️  {arquivo} (sem coluna 'Discriminação' — ignorado)")
                continue
        except Exception:
            print(f"     ⚠️  {arquivo} (não foi possível ler — ignorado)")
            continue

        arquivos_encontrados.append((caminho_arquivo, item, arquivo))
        print(f"     ✅ {arquivo} (válido)")

if not arquivos_encontrados:
    print("\n❌ Nenhuma planilha válida encontrada.")
    raise SystemExit()

print(f"\n✅ {len(arquivos_encontrados)} planilha(s) válida(s) encontrada(s).")

# --- Senha de app do Gmail ---
# Não fica salva no notebook: vem dos Secrets do Colab (nome SENHA_APP)
# ou é digitada numa caixa que esconde o texto.
def obter_senha_app():
    try:
        from google.colab import userdata
        senha = userdata.get('SENHA_APP')
        if senha:
            print("\n🔑 Senha de app lida dos Secrets do Colab.")
            return senha
    except Exception:
        pass
    return getpass.getpass("\n🔑 Digite a senha de app do Gmail e tecle Enter (ela não fica salva): ")

# A senha de app aparece em grupos ("abcd efgh ijkl mnop"); os espaços são só visuais.
SENHA_APP = (obter_senha_app() or "").replace(" ", "").strip()

# --- Regras de classificação ---
def singular(palavra):
    """Leva plurais comuns ao singular: consultas → consulta, exames → exame,
    hospitalares → hospitalar, produções → producao, hospitais → hospital."""
    if len(palavra) <= 3:
        return palavra
    for fim, troca in (("oes", "ao"), ("ais", "al"), ("eis", "el"), ("ois", "ol"),
                       ("res", "r"), ("zes", "z"), ("ns", "m")):
        if palavra.endswith(fim):
            return palavra[:-len(fim)] + troca
    if palavra.endswith("s"):
        return palavra[:-1]
    return palavra

def limpar_texto(texto):
    if pd.isna(texto):
        return ""
    texto_str = str(texto).lower()
    texto_str = unicodedata.normalize('NFKD', texto_str).encode('ASCII', 'ignore').decode('utf-8')
    # Pontuação, espaços repetidos e quebras de linha viram um espaço só;
    # cada palavra vai para o singular, para que o plural caia na mesma regra.
    texto_str = re.sub(r'[^a-z0-9]+', ' ', texto_str)
    return " ".join(singular(p) for p in texto_str.split())

# Termos escritos como na regra do cliente. Eles passam pela mesma
# limpeza do texto, então cada termo já cobre o plural.
TERMOS_CONSULTA = [
    'consulta médica',
    'realizadas em coordenação',     # "coordenação" sozinha NÃO basta
    'prestação de serviços médicos',
    'prestacao ded servicos',        # erro de digitação frequente nas planilhas
    'serviços médicos prestados',
    'folha médica',
]
TERMOS_PROCEDIMENTO = [
    'procedimento',
    'cirurgia',
    'exame diagnóstico',
    'serviço hospitalar',
    'ambulatório',
    'hospital',
    'produção',
]
TERMOS_ANALISAR = [
    'atendimento',
]

TERMOS_CONSULTA_LIMPOS = [limpar_texto(t) for t in TERMOS_CONSULTA]
TERMOS_PROCEDIMENTO_LIMPOS = [limpar_texto(t) for t in TERMOS_PROCEDIMENTO]
TERMOS_ANALISAR_LIMPOS = [limpar_texto(t) for t in TERMOS_ANALISAR]

def classificar_servico(descricao):
    desc = limpar_texto(descricao)

    tem_consulta     = any(termo in desc for termo in TERMOS_CONSULTA_LIMPOS)
    tem_procedimento = any(termo in desc for termo in TERMOS_PROCEDIMENTO_LIMPOS)
    tem_atendimento  = any(termo in desc for termo in TERMOS_ANALISAR_LIMPOS)

    # Prioridade: ANALISAR > CONSULTA > PROCEDIMENTO
    if tem_atendimento or (tem_consulta and tem_procedimento):
        return 'ANALISAR'
    elif tem_consulta:
        return 'CONSULTA'
    elif tem_procedimento:
        return 'PROCEDIMENTO'
    else:
        return ''

# --- Gravar a coluna nova sem mexer no resto da planilha ---
def gravar_planilha_classificada(caminho_origem, caminho_destino):
    """Abre a planilha original e só acrescenta a coluna 'Classificação' na
    primeira aba. Outras abas, fórmulas, estilos, formatos de data e moeda,
    larguras, células mescladas e painéis congelados ficam como estavam.
    Devolve a lista de classificações das linhas com dados."""
    wb = openpyxl.load_workbook(caminho_origem)                     # mantém fórmulas e formatação
    ws = wb.worksheets[0]                                           # a mesma aba que o filtro leu
    ws_valores = openpyxl.load_workbook(caminho_origem, data_only=True).worksheets[0]

    cabecalho = [c.value for c in ws[1]]
    col_desc = cabecalho.index('Discriminação') + 1
    if 'Classificação' in cabecalho:
        col_class = cabecalho.index('Classificação') + 1            # arquivo já classificado: atualiza
    else:
        col_class = ws.max_column + 1

    vizinha = col_class - 1
    celula = ws.cell(row=1, column=col_class)
    if vizinha >= 1 and ws.cell(row=1, column=vizinha).has_style:
        celula._style = copy(ws.cell(row=1, column=vizinha)._style)
    celula.value = 'Classificação'

    classificacoes = []
    for linha in range(2, ws_valores.max_row + 1):
        valores = [c.value for c in ws_valores[linha]]
        if all(v is None or str(v).strip() == "" for v in valores):
            continue                                                # linha totalmente vazia
        resultado = classificar_servico(ws_valores.cell(row=linha, column=col_desc).value)
        classificacoes.append(resultado)

        celula = ws.cell(row=linha, column=col_class)
        if vizinha >= 1 and ws.cell(row=linha, column=vizinha).has_style:
            celula._style = copy(ws.cell(row=linha, column=vizinha)._style)
            celula.number_format = 'General'
        celula.value = resultado or None

    letra = get_column_letter(col_class)
    if ws.column_dimensions[letra].width is None or ws.column_dimensions[letra].width < 16:
        ws.column_dimensions[letra].width = 16

    # Se a planilha tem filtro no cabeçalho, a coluna nova entra no filtro
    if ws.auto_filter.ref:
        min_col, min_row, max_col, max_row = range_boundaries(ws.auto_filter.ref)
        if max_col == col_class - 1:
            ws.auto_filter.ref = (f"{get_column_letter(min_col)}{min_row}:"
                                  f"{letra}{max_row}")

    wb.save(caminho_destino)
    return classificacoes

# --- Processar cada planilha ---
arquivos_gerados = []
resumo_geral     = []
erros            = []

print("\n🔄 Processando planilhas...\n")

for i, (caminho_arquivo, nome_subpasta, nome_arquivo) in enumerate(arquivos_encontrados, 1):
    print(f"  [{i}/{len(arquivos_encontrados)}] {nome_subpasta}/{nome_arquivo}...", end=" ")

    try:
        nome_saida    = f"CLASSIFICADO_{nome_subpasta}_{nome_arquivo}"
        caminho_saida = f"/content/{nome_saida}"
        classificacoes = gravar_planilha_classificada(caminho_arquivo, caminho_saida)

        arquivos_gerados.append(caminho_saida)

        contagem     = pd.Series(classificacoes, dtype=object).value_counts().to_dict()
        vazias       = contagem.get('', 0)
        linha_resumo = (
            f"• {nome_subpasta}/{nome_arquivo}: "
            f"CONSULTA={contagem.get('CONSULTA',0)} | "
            f"PROCEDIMENTO={contagem.get('PROCEDIMENTO',0)} | "
            f"ANALISAR={contagem.get('ANALISAR',0)}"
        )
        if vazias > 0:
            linha_resumo += f" | SEM CATEGORIA={vazias} ⚠️"

        resumo_geral.append(linha_resumo)
        print("✅")

    except Exception as e:
        print(f"❌ erro: {e}")
        erros.append(f"• {nome_subpasta}/{nome_arquivo}: {e}")

# --- Enviar e-mail ---
if not arquivos_gerados:
    print("\n❌ Nenhuma planilha foi processada. E-mail não enviado.")
elif not SENHA_APP:
    print("\n❌ Senha de app não informada. E-mail não enviado.")
    print("Rode esta célula de novo e digite a senha quando ela for pedida.")
else:
    print(f"\n📧 Enviando e-mail com {len(arquivos_gerados)} anexo(s)...")

    aviso_erros = ""
    if erros:
        aviso_erros = "\n\n⚠️ Arquivos com problema (não incluídos):\n" + "\n".join(erros)

    corpo = (
        f"Olá,\n\n"
        f"Seguem em anexo as {len(arquivos_gerados)} planilha(s) classificadas.\n\n"
        f"📊 Resumo por arquivo:\n"
        + "\n".join(resumo_geral)
        + aviso_erros +
        f"\n\nAtt,\nSistema de Classificação"
    )

    msg = EmailMessage()
    msg['Subject'] = f'Planilhas Classificadas — {len(arquivos_gerados)} arquivo(s) — Pasta: {PASTA_DRIVE}'
    msg['From']    = EMAIL_REMETENTE
    msg['To']      = EMAIL_DESTINATARIO
    msg.set_content(corpo)

    for caminho_saida in arquivos_gerados:
        with open(caminho_saida, 'rb') as f:
            msg.add_attachment(
                f.read(),
                maintype='application',
                subtype='vnd.openxmlformats-officedocument.spreadsheetml.sheet',
                filename=os.path.basename(caminho_saida)
            )

    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login(EMAIL_REMETENTE, SENHA_APP)
            smtp.send_message(msg)
        print(f"✅ E-mail enviado para {EMAIL_DESTINATARIO}!")
    except Exception as e:
        print(f"❌ Erro ao enviar e-mail: {e}")
        print("Verifique se o e-mail e a senha de app estão corretos.")

# --- Resultado final ---
print("\n" + "=" * 50)
print(f"  CONCLUÍDO: {len(arquivos_gerados)}/{len(arquivos_encontrados)} planilhas processadas")
if erros:
    print(f"  ATENÇÃO: {len(erros)} arquivo(s) com problema")
print("=" * 50)